# 01 — Scrape and inventory source data

This notebook acquires the raw inputs used by the injury pipeline. Read [`context/SCRAPING_CONTEXT.md`](../context/SCRAPING_CONTEXT.md) before running it. Network actions are disabled by default so opening the notebook cannot accidentally send hundreds of requests.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
SRC_DIR = PROJECT_ROOT / 'src'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT

## Source inventory

Prosports supplies medical wording and injury dates; NBA and Basketball-Reference game data supply availability; Basketball-Reference advanced tables supply VORP. The Kaggle file is a Prosports-derived archive, not an independent medical source.

In [ ]:
sources = pd.DataFrame([
    {'Source': 'Pro Sports Transactions', 'URL': 'https://www.prosportstransactions.com/basketball/Search/Search.php', 'Role': 'Injury dates and raw Notes'},
    {'Source': 'Official NBA team box scores', 'URL': 'https://www.nba.com/stats/teams/boxscores', 'Role': 'Game dates and actual player minutes'},
    {'Source': 'Basketball-Reference', 'URL': 'https://www.basketball-reference.com/', 'Role': 'Schedules, game-log audits, biographies, and VORP'},
    {'Source': 'Kaggle Prosports archive', 'URL': 'https://www.kaggle.com/datasets/loganlauton/nba-injury-stats-1951-2023', 'Role': 'Historical archive and cross-check'},
])
sources

## Retrieval controls

Set only the jobs you intend to run. Prosports may return HTTP 403; a failed request must remain visible and must not be treated as complete coverage. Place the manually downloaded Kaggle archive at `data/raw/kaggle_nba_injury_stats_1951_2023.csv`.

In [ ]:
RUN_PROSPORTS = False
RUN_HISTORICAL_SCHEDULES = False
RUN_VORP = False
BEGIN_DATE = '2000-01-01'
END_DATE = '2026-07-26'

In [ ]:
def run_script(name, *args):
    command = [sys.executable, str(SRC_DIR / name), *map(str, args)]
    print(' '.join(command))
    return subprocess.run(command, cwd=PROJECT_ROOT, check=True)

if RUN_PROSPORTS:
    # build_dataset.py exposes the tested Prosports pagination scraper.
    # It also rebuilds the processed table after retrieval.
    run_script('build_dataset.py', '--scrape', '--begin-date', BEGIN_DATE, '--end-date', END_DATE)

if RUN_HISTORICAL_SCHEDULES:
    run_script('download_historical_schedules.py')

if RUN_VORP:
    run_script('ingest_vorp.py')

## Raw-file audit

The inventory below is deliberately source-agnostic. It records file size, rows, columns, date coverage, and exact duplicate rows before any cleaning.

In [ ]:
def inventory_csv(path):
    try:
        frame = pd.read_csv(path)
    except Exception as exc:
        return {'File': path.name, 'Status': f'READ ERROR: {exc}'}
    date_col = next((c for c in ['Date', 'date', 'GAME_DATE'] if c in frame.columns), None)
    dates = pd.to_datetime(frame[date_col], errors='coerce') if date_col else pd.Series(dtype='datetime64[ns]')
    return {
        'File': path.name,
        'Status': 'OK',
        'Rows': len(frame),
        'Columns': len(frame.columns),
        'Exact duplicates': int(frame.duplicated().sum()),
        'First date': dates.min().date() if not dates.empty and dates.notna().any() else None,
        'Last date': dates.max().date() if not dates.empty and dates.notna().any() else None,
    }

raw_inventory = pd.DataFrame(inventory_csv(path) for path in sorted(RAW_DIR.glob('*.csv')))
raw_inventory if not raw_inventory.empty else 'No raw CSVs yet—run a selected retrieval job or add the archive files.'

In [ ]:
required_for_core_build = [
    'kaggle_nba_injury_stats_1951_2023.csv',
    'prosportstransactions_scrape_missedgames_2010_2019.csv',
    'prosportstransactions_scrape_IRL_2010_2019.csv',
    'all_teams_schedule_2010_2020.csv',
]
availability = pd.DataFrame({
    'Required raw file': required_for_core_build,
    'Present': [(RAW_DIR / name).exists() for name in required_for_core_build],
})
availability